# HACKBLR Project - Data Processing and Imputation


In [24]:
# Dependencies
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import os
from datetime import datetime

In [25]:
# Loading the source CSV file
file_path = "/content/CIP_LATEST.csv"
df = pd.read_csv(file_path)

In [26]:
# 2. Handle Skip-Logic and Initial Cleaning
skip_logic_cols = [
    'Other_illness', 'Period_of_non_consumption', 'Intoxicant',
    'Intoxicant_expenses', 'intoxicants_caused_conflicts',
    'Reason_for_Intoxicant', 'Substance_Used', 'Usage_Frequency'
]
df[skip_logic_cols] = df[skip_logic_cols].fillna(0)

# Separate text columns and convert remaining features to numeric
text_cols = ['Name', 'SUBTRIBE']
df_text = df[text_cols].copy()
df_numeric = df.drop(columns=text_cols).apply(pd.to_numeric, errors='coerce')

print("Base cleaning completed. Data ready for splitting.")

Base cleaning completed. Data ready for splitting.


In [27]:
# Dataset Partitioning (Original vs Missing Data)

# Identify records with missing values
missing_mask = df_numeric.isna().any(axis=1)

# Partition the data into complete and missing sets
df_complete = df_numeric[~missing_mask].copy()
df_missing = df_numeric[missing_mask].copy()

print(f"Total records: {len(df_numeric)}")
print(f"Complete records (Training set): {len(df_complete)}")
print(f"Records with missing data (Prediction set): {len(df_missing)}")

# Note: Create a tracking column to differentiate records for later analysis
df_numeric['Was_Missing'] = missing_mask

Total rows: 200
Rows with NO missing data (Pristine/Train): 200
Rows WITH missing data (To be predicted): 0


In [28]:
# Random Forest Iterative Imputation

# Remove tracking column before imputation
X_to_impute = df_numeric.drop(columns=['Was_Missing'])

print("Training Random Forest imputer to estimate missing values based on complete records...")
rf_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=50, random_state=42),
    max_iter=10,
    random_state=42
)

# Perform imputation
imputed_array = rf_imputer.fit_transform(X_to_impute)
df_imputed = pd.DataFrame(imputed_array, columns=X_to_impute.columns).round().astype(int)

# Reattach the tracking column
df_imputed['Was_Missing'] = df_numeric['Was_Missing'].values

Training Random Forest to learn from pristine rows and predict missing ones...


In [29]:
# Comparison Analysis: e.g., 'NonMed_Expense'
# (If 'NonMed_Expense' contains no missing values, select an alternative column for comparison)

# Statistics for original complete records
original_stats = df_complete['NonMed_Expense'].describe()

# Statistics for imputed records only
imputed_rows_only = df_imputed[df_imputed['Was_Missing'] == True]
predicted_stats = imputed_rows_only['NonMed_Expense'].describe()

print("--- COMPARISON: NonMed_Expense ---")
print("1. ORIGINAL COMPLETE RECORDS (Actual Data):")
print(f"   Mean Expense: ₹{original_stats['mean']:.2f}")
print(f"   Max Expense:  ₹{original_stats['max']:.2f}\n")

print("2. ML PREDICTED RECORDS (Imputed Data):")
print(f"   Mean Expense: ₹{predicted_stats['mean']:.2f}")
print(f"   Max Expense:  ₹{predicted_stats['max']:.2f}\n")

# Display a sample of imputed patient records
print("Sample of imputed records predicted by ML:")
display(df_imputed[df_imputed['Was_Missing'] == True].head(3))

--- COMPARISON: NonMed_Expense ---
1. ORIGINAL COMPLETE ROWS (Actual Data):
   Mean Expense: ₹7585.01
   Max Expense:  ₹200000.00

2. ML PREDICTED ROWS (Imputed Data):
   Mean Expense: ₹nan
   Max Expense:  ₹nan

Sample of patients whose data was predicted by ML:


,Gender,Age_yrs,UHID,CRF,Dis_from_CIP_km,Age_Onset_days,Duration_days,First_Contact,NonMed_Expense,Time_to_Doc,...,CAMI_Q03,CAMI_Q04,CAMI_Q05,CAMI_Q06,CAMI_Q07,CAMI_Q08,CAMI_Q09,CAMI_Q10,CAMI_Q11,Was_Missing


In [30]:
# Final Data Consolidation and Export

# Remove tracking column
df_imputed = df_imputed.drop(columns=['Was_Missing'])

# Reconstruct final dataframe by merging text and numeric features
df_final = pd.concat([df_text, df_imputed], axis=1)

# Partition data for downstream machine learning tasks
# Define 'First_Contact' as the target variable for prediction
X = df_final.drop(columns=['Name', 'SUBTRIBE', 'First_Contact'])
y = df_final['First_Contact']

# Perform an 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Final Data Partitioning -> Training Features: {X_train.shape}, Testing Features: {X_test.shape}")

# Save processed data for Vector Database injection

# 1. Define base filename with current date
today_str = datetime.now().strftime("%Y%m%d")
base_filename = f"Raw_Data_v{today_str}"
extension = ".csv"

# 2. Determine next available version number for the current date
version = 1
while os.path.exists(f"{base_filename}_{version}{extension}"):
    version += 1

# 3. Construct final filename
final_filename = f"{base_filename}_{version}{extension}"

# 4. Export to CSV
df_final.to_csv(final_filename, index=False)
print(f"Processed data saved as: {final_filename}")

Final Data shapes -> Training Features: (160, 82), Testing Features: (40, 82)
Data saved as: Raw_Data_v20260415_2.csv
